In [1]:
import numpy as np, json

data = json.load(open("response_1777349958303.json"))
arr = np.array(json.loads(data["result"]["result"]))
print(arr.shape)

(1, 1, 6, 8400)


In [2]:
arr = arr.squeeze()
arr = arr.transpose()
print(arr[0])

[9.00840092e+00 1.51822586e+01 1.87367802e+01 4.23507233e+01
 7.77840614e-06 6.25848770e-06]


In [3]:
scores = arr[:, 4:]
max_scores = scores.max(axis=1)
print(max_scores.max())
print((max_scores > 0.25).sum())

0.9272082448005676
10


In [4]:
keep = max_scores > 0.25
boxes = arr[keep]
scores = max_scores[keep]
order = scores.argsort()[::-1]
for i in order:
    print(f"score={scores[i]:.3f} box={boxes[i, :4]}  cls={boxes[i, 4:].argmax()}")

score=0.927 box=[461.53875732 445.62930298 125.46524048 164.84857178]  cls=0
score=0.915 box=[461.12731934 446.30712891 125.20776367 164.61453247]  cls=0
score=0.909 box=[461.85470581 446.28924561 126.21453857 166.28451538]  cls=0
score=0.903 box=[461.76431274 445.68103027 125.65240479 165.33566284]  cls=0
score=0.899 box=[461.23065186 446.27618408 126.31799316 165.08639526]  cls=0
score=0.895 box=[461.9654541  445.81045532 126.12600708 165.35186768]  cls=0
score=0.883 box=[461.35638428 445.53668213 125.56869507 164.60049438]  cls=0
score=0.864 box=[461.79882812 446.01644897 126.02612305 164.91204834]  cls=0
score=0.840 box=[460.2645874  446.06234741 125.45666504 163.82818604]  cls=0
score=0.816 box=[461.35971069 446.06225586 126.8258667  164.36901855]  cls=0


In [5]:
x1, y1 = boxes[:, 0] - (boxes[:, 2]/2), boxes[:, 1] - (boxes[:, 3]/2)
x2, y2 = boxes[:, 0] + (boxes[:, 2]/2), boxes[:, 1] + (boxes[:, 3]/2)
box_xyxy = np.stack((x1, y1, x2, y2), axis=1)

def nms_numpy(box_xyxy, scores, iou_threshold=0.5):
    x1 = box_xyxy[:, 0]
    y1 = box_xyxy[:, 1]
    x2 = box_xyxy[:, 2]
    y2 = box_xyxy[:, 3]
    
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h

        iou = inter / (areas[i] + areas[order[1:]] - inter)

        inds = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]
    return keep

In [6]:
result = nms_numpy(box_xyxy, scores, iou_threshold=0.5)
box_xyxy_kept = np.stack([x1[result], y1[result], x2[result], y2[result]], axis=1)
print(result)

[np.int64(0)]


In [7]:
score, cls = scores[result], boxes[result, 4:].argmax(axis=1)
print(score.shape, box_xyxy_kept.shape, cls.shape)

(1,) (1, 4) (1,)
